# ECG Classification & SLoC Interpretability


# Setup & Imports

In [ ]:
%pip install neurokit2
%pip install wfdb
%pip install pandas==2.3.3

import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from dataset import load_data, split_folds, create_dataset, get_weighting
from models.cct import TransformerECG as CCT
from models.cnn import ConvNet as CNN
from models.cnnbilstm import CNNBiLSTM as LSTM
from training import fit, optimise_attribution
from metrics import display_metrics, deletion_insertion_auc, gen_avgs, analyse_saliency, std_per_label, cross_label_discriminability
from utils import (seed_everything, get_predictions, plot_loss_history, plot_sloc_loss_history,
                   overlay_attr_on_grid, print_report, plot_results,
                   plot_discriminability, iii_heatmap_comparison, filter_nan)

# Data Loading & Preprocessing

In [ ]:
DATA_PATH = 'data/raw/ptb-xl-100hz/'
WEIGHTS_PATH = 'data/weights/'
RESULTS_PATH = 'data/processed/'

# Data too big too put on github
X, Y = load_data(DATA_PATH)

folds = split_folds(X, Y)
X_train, y_train = folds['train']
X_valid, y_valid = folds['valid']
X_test, y_test = folds['test']

print(f"Total samples: {len(X)}")

In [ ]:
clean = False
split = True

train_removed, train_dataset = create_dataset(X_train, y_train, clean, split)
valid_removed, valid_dataset = create_dataset(X_valid, y_valid, clean, split)
test_removed, test_dataset = create_dataset(X_test, y_test, clean, split)

print(f"Removed: train={len(train_removed)}, valid={len(valid_removed)}, test={len(test_removed)}")

weighting = get_weighting(train_dataset.tensors[1])
print(f"Class weights: {weighting}")

# Model Setup

In [ ]:
model_type = 'cct'  # 'cct', 'cnn', or 'lstm'
train = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
torch.cuda.empty_cache()

labels = ['NORM', 'HYP', 'MI', 'CD', 'STTC']
batch_size = 128
num_epochs = 50

if model_type == 'cct':
    model = CCT(num_leads=12, embed_dim=100, patch_size=5, num_heads=5,
                depth=6, mlp_dim=256, dropout=0.4, device=device, max_seq_len=1000)
    learning_rate = 0.00025
    regularisation = None
    weight_file = 'cct_clean_final.pth'

elif model_type == 'cnn':
    model = CNN(padding='same', drop_p=0.3, pool_size=2)
    learning_rate = 0.0005
    regularisation = 'L2'

    # CNN weights too big for github also
    weight_file = None

elif model_type == 'lstm':
    model = LSTM(padding='same', drop_p=0.3, pool_size=2, lstm_hidden=128, lstm_layers=1)
    learning_rate = 0.0005
    regularisation = 'L2'
    weight_file = 'lstm_clean.pth'

model = model.to(device)
print(f"Model: {model_type.upper()}")

# Training / Loading Weights

In [ ]:
if train:
    if model_type == 'cct':
        optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    else:
        optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model, loss_hist_train, loss_hist_valid, best_loss, best_epoch = fit(
        train_dataset, valid_dataset, weighting, model, optimiser,
        batch_size, regularisation, device, num_epochs=num_epochs
    )
    print(f"Best validation loss: {best_loss:.4f} at epoch {best_epoch}")
    plot_loss_history(loss_hist_train, loss_hist_valid)
else:
    assert weight_file, f"No pretrained weights for {model_type}. Set train=True."
    weight_path = os.path.join(WEIGHTS_PATH, weight_file)

    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()

    print(f"Loaded weights from {weight_path}")

# Model Evaluation

In [ ]:
valid_loader = DataLoader(valid_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

v_metrics = display_metrics(get_predictions(model, valid_loader, device), "Validation", labels)
t_metrics = display_metrics(get_predictions(model, test_loader, device), "Test", labels)

# SLoC - Single Sample Pipeline

In [ ]:
sloc_hyperparams = {
    "nmasks": 6400, 
    "batch_size": 16, 
    "segsize": 10, 
    "prob": 0.7,
    "epochs": 200, 
    "lr": 0.1, 
    "tv_eps": 0.1, 
    "l1_eps": 0.01, 
    "norm": True
}

figures = []

sample = random.randint(0,4316)
input = test_dataset[sample][0]
pred_logits = model(input.unsqueeze(0).to(device)).detach().cpu()
pred = np.argmax(pred_logits.numpy())
truth = test_dataset[sample][1]
score = F.softmax(pred_logits, dim=1)[:,pred]


print(f"Sample: {sample}\nLabel: {pred}\nConfidence: {float(score[0])*100}%\nTruth: {truth}")

attribution, loss_hist = optimise_attribution(device, input, model,
    sloc_hyperparams["nmasks"], sloc_hyperparams["batch_size"],
    sloc_hyperparams["segsize"], sloc_hyperparams["prob"],
    sloc_hyperparams["epochs"], sloc_hyperparams["lr"],
    sloc_hyperparams["tv_eps"], sloc_hyperparams["l1_eps"],
    sloc_hyperparams["norm"], label=truth
)

attr_np = attribution.attr_map.detach().cpu().numpy()
signals_np = input.detach().cpu().numpy()
figures.append(overlay_attr_on_grid(signals_np, attr_np, cmap="hot", alpha=0.6, clip_percentile=(1, 99)))

plot_sloc_loss_history(loss_hist)

### Show Heatmap

In [ ]:
attr_np = attribution.attr_map.detach().cpu().numpy()
signals_np = input.detach().cpu().numpy()
overlay_attr_on_grid(signals_np, attr_np, cmap="hot", alpha=0.6, clip_percentile=(1, 99))

### INS / DEL Metrics

In [ ]:
res = deletion_insertion_auc(model=model, input=input, attr=attr_np, device=device,
                             target_class=truth, steps=100, batch_size=32)

print(f"INS AUC: {res['ins_auc']:.4f}  |  DEL AUC: {res['del_auc']:.4f}")

plt.plot(res["fractions"], res["ins_curve"], label='Insertion')
plt.plot(res["fractions"], res["del_curve"], label='Deletion')
plt.legend()
plt.show()

# SLoC - Aggregate Testing

In [ ]:
del_ins_params = {"steps": 100, "batch_size": 512}

for label_idx in range(5):
    bool_correct = t_metrics['preds'] == t_metrics['true']
    bool_label = t_metrics['true'] == label_idx
    bool_arr = bool_label * bool_correct

    label_set = test_dataset[bool_arr]

    if label_idx == 1:  # HYP — merge valid + test for more samples
        v_correct = v_metrics['preds'] == v_metrics['true']
        v_label = v_metrics['true'] == label_idx
        v_set = valid_dataset[v_label * v_correct]
        merged = [torch.cat((v_set[0], label_set[0])), torch.cat((v_set[1], label_set[1]))]
        label_set2 = TensorDataset(merged[0][:200], merged[1][:200])
    else:
        label_set2 = TensorDataset(label_set[0][:200], label_set[1][:200])

    result = gen_avgs(device, model, label_set2, sloc_hyperparams, del_ins_params, log=True)
    np.save(f'{model_type}-sloc-results-{label_idx}', result)
    print(f"Label {labels[label_idx]}: INS={result['insauc_avg']:.4f} DEL={result['delauc_avg']:.4f}")

# SLoC - Results Analysis

In [ ]:
label_names = ["NORM", "HYP", "MI", "CD", "STTC"]

# Load saved results
results_list = []
all_atts = []
for i in range(5):
    r = np.load(os.path.join(RESULTS_PATH, f"{model_type}-sloc-results",
                             f"{model_type}-sloc-results-{i}.npy"), allow_pickle=True)[()]
    results_list.append(r)
    att = r['attributions']
    if len(att) < 200:
        pad = np.full((200 - len(att), 12, 500), np.nan)
        att = np.concatenate((att, pad))
    all_atts.append(att)

attr_maps = np.array(all_atts)

result = analyse_saliency(attr_maps, sp_size=5, label_names=label_names)
print_report(result)
plot_results(result)

disc_result = cross_label_discriminability(result)
plot_discriminability(result, disc_result)
iii_heatmap_comparison(result)